![Module 3: Self-Modifying Prompt](../images/module-3-self-modify.svg)

# Module 3: Self-Modifying Prompt

> The agent rewrites its own system prompt. Teach it once ("respond in haiku"); it persists to disk and survives restarts. **Behavior change, not just context.**

📖 Full explainer: see the companion page in `content/` of this repo.  
💻 Standalone script: `code/step-02-system-prompt/agent.py`  
🔗 Workshop: https://github.com/strands-agents/samples/tree/main/python/01-learn/18-self-improving-agents

---


# AIM308 — Module 2: Self-Modifying System Prompt

In [ ]:
%pip install -q strands-agents strands-agents-tools

In [ ]:
import os
import boto3
ssm = boto3.client('ssm')
MODEL_ID = ssm.get_parameter(Name='/aim308/model-id')['Parameter']['Value']  # Claude Sonnet 4.5
REGION = ssm.get_parameter(Name='/aim308/region')['Parameter']['Value']
os.environ["AWS_REGION"] = REGION
os.environ["BYPASS_TOOL_CONSENT"] = os.getenv("BYPASS_TOOL_CONSENT", "true")
os.environ["STRANDS_TOOL_CONSOLE_MODE"] = "enabled"

## The `system_prompt` tool — agent writes to its own env var and a `.prompt` file

In [ ]:
from pathlib import Path
from strands import tool
PROMPT_FILE = Path(".prompt")

@tool
def system_prompt(action: str, prompt: str | None = None) -> dict:
    """Manage the agent's own system prompt.

    Args:
        action: One of "view", "update", or "reset".
            - "view"   -> return the current persisted prompt.
            - "update" -> persist `prompt` to env + .prompt file (aliases: "set", "write", "save").
            - "reset"  -> clear the persisted prompt (aliases: "clear", "delete").
        prompt: New prompt text. Required when action is "update".
    """
    action = (action or "").lower()
    if action in ("update", "set", "write", "save"):
        if not prompt:
            return {"status": "error", "content": [{"text": "update requires a 'prompt' value"}]}
        os.environ["SYSTEM_PROMPT"] = prompt
        PROMPT_FILE.write_text(prompt)
        return {"status": "success", "content": [{"text": "Prompt updated & persisted."}]}
    if action in ("view", "get", "show"):
        return {"status": "success", "content": [{"text": os.environ.get("SYSTEM_PROMPT", "(empty)")}]}
    if action in ("reset", "clear", "delete"):
        os.environ.pop("SYSTEM_PROMPT", None)
        if PROMPT_FILE.exists():
            PROMPT_FILE.unlink()
        return {"status": "success", "content": [{"text": "Reset."}]}
    return {"status": "error", "content": [{"text": f"Unknown action: {action!r}. Use view, update, or reset."}]}

## Dynamic system-prompt construction

In [ ]:
from datetime import datetime

def build_system_prompt():
    base = "You are a self-improving research agent. You can modify your own system prompt."
    persistent = PROMPT_FILE.read_text() if PROMPT_FILE.exists() else ""
    env_ext = os.environ.get("SYSTEM_PROMPT", "")
    runtime = f"Time: {datetime.now().isoformat(timespec='seconds')}"
    return "\n".join(filter(None, [base, persistent, env_ext, runtime]))

## Build the agent

In [ ]:
from strands import Agent
from strands_tools import shell
def new_agent():
    return Agent(
        model=MODEL_ID,
        tools=[shell, system_prompt],
        system_prompt=build_system_prompt(),
    )

## Teach it to respond in haiku

In [ ]:
new_agent()("view your system prompt")

In [ ]:
new_agent()("from now on always reply in haiku — update your prompt permanently")

In [ ]:
new_agent()("what's 2+2?")  # should be a haiku

## Reset

In [ ]:
new_agent()("reset your system prompt")

Next: **[03_memory.ipynb](03_memory.ipynb)**